In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [7]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt(jahid_url))


Here is the list of links on the website https://hellojahid.github.io/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://hellojahid.github.io/assets/CV of Jahid.pdf
https://www.rmit.edu.au/
https://www.carsales.com.au/
https://www.rmit.edu.au/contact/staff-contacts/academic-staff/o/ong---kok-leong
https://www.linkedin.com/in/agustinus-nalwan/
https://preneurlab.ca/
https://www.youtube.com/@MachineLearningTalksTips
https://www.youtube.com/channel/UCwCItw_xlM9aN_O6U8S-DtQ
https://www.researchgate.net/profile/Md-Jahid-Hasan-2
https://scholar.google.com/citations?user=7gQWnDMAAAAJ&hl=en
https://doi.org/10.1002/widm.70027
https://doi.org/10.1109/ACCESS.2024.3506563
https://ieeexplore.ieee.org/abstract/document/9396878
https://doi.org/10.1109/EICT48899.2019.9068817
https://doi.org/10.1109/ICBSLP47725.201

In [14]:

def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [15]:
select_relevant_links(jahid_url)

Selecting relevant links for https://hellojahid.github.io/ by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'about page', 'url': 'https://hellojahid.github.io/'},
  {'type': 'product page', 'url': 'https://hellojahid.github.io/app.html'},
  {'type': 'portfolio page',
   'url': 'https://hellojahid.github.io/memories.html'},
  {'type': 'CV / resume',
   'url': 'https://hellojahid.github.io/assets/CV%20of%20Jahid.pdf'},
  {'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/in/hellojahid/'},
  {'type': 'GitHub profile', 'url': 'https://github.com/hellojahid/'},
  {'type': 'partner page', 'url': 'https://preneurlab.ca/'}]}

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("csiro", "https://www.csiro.au/")

Selecting relevant links for https://www.csiro.au/ by calling gpt-5-nano
Found 19 relevant links


# CSIRO: Australia's National Science Agency

---

## Who We Are  
The Commonwealth Scientific and Industrial Research Organisation (CSIRO) is Australia's national science agency and innovation catalyst. For nearly a century, CSIRO has been dedicated to solving the greatest challenges through cutting-edge science and technology, improving lives across Australia and beyond.

---

## Our Purpose  
At CSIRO, we apply research across a wide range of scientific disciplines throughout the innovation lifecycle. We inform policy, develop new industries, and help evolve existing sectors to drive economic, environmental, and social progress. Our work is focused on impactful innovation that contributes solutions to some of Australia’s and the world’s most pressing challenges.

---

## What We Do  
- **Innovative Science & Technology:** From genomics to climate science, our research helps decode biodiversity, understand climate events like heatwaves, and harness technology for social good.  
- **Industry Collaboration:** We offer innovation, funding, and entrepreneurship programs to connect researchers with industry, enabling impactful projects and commercial developments.  
- **Policy Development:** Our science informs public policy at national and international levels, ensuring evidence-based decision-making.  
- **Indigenous Engagement:** CSIRO acknowledges and respects the Traditional Owners of the lands where we work. We support Indigenous-led science projects and recognise the vital contributions of Aboriginal and Torres Strait Islander peoples to Australian culture, economy, and science.

---

## Commitment to Culture & Inclusion  
CSIRO fosters an inclusive work environment that honours diversity, Indigenous heritage, and community contributions. We openly acknowledge the Traditional Custodians of Australia's lands, seas, and waters and actively promote Indigenous science initiatives. Our culture encourages collaboration, innovation, and respect.

---

## Customers & Impact  
Our clients and partners span government, industry, research institutions, and Indigenous enterprises seeking solutions for complex problems such as:  
- Climate change and environmental management  
- Health crises including COVID-19 research  
- Agricultural innovations and food security  
- Sustainable resource use and biodiversity conservation  

We empower businesses to unlock success through science and technology, advancing Australian industries in a competitive global landscape.

---

## Careers at CSIRO  
CSIRO offers exciting career opportunities for scientists, researchers, technologists, and innovators passionate about creating real-world impact. Joining CSIRO means becoming part of a dynamic team that values diverse perspectives and fosters continuous learning through:  
- Cutting-edge research projects  
- Industry collaborations and entrepreneurship programs  
- Support for Indigenous and community-focused initiatives  

Explore our career paths and help shape the future through science.

---

## Connect with CSIRO  
Learn more about how CSIRO can work with your business or community, or explore opportunities to join our team through our website.

**Website:** [csiro.au](https://www.csiro.au)  
**Explore:** Research, Innovation Programs, Careers, Education, News & Events

---

CSIRO – Solving today’s problems for a better tomorrow through innovation, science, and partnership.

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the vibrant **AI community building the future** of machine learning. Serving as the central hub for ML enthusiasts, engineers, scientists, and organizations, Hugging Face offers an all-encompassing platform to **collaborate, create, and innovate** with machine learning models, datasets, and applications. 

At its core, Hugging Face empowers the next generation of machine learning practitioners to share and experiment with **open-source models and data** in pursuit of building an **open and ethical AI future**.

---

## What We Offer

- **2M+ Models:** Discover and deploy the most advanced models across text, image, video, audio, and even 3D modalities.
- **500k+ Datasets:** Access a rich library of datasets to train and evaluate AI models.
- **1M+ Applications:** Explore AI applications and demos built by the community and create your own portfolio of ML projects.
- **Spaces:** Host and share interactive ML apps easily with built-in compute.

Our open-source HF stack accelerates your AI innovation, whether you are exploring new research or building scalable production solutions.

---

## Enterprise Solutions

Hugging Face enables organizations to scale their AI efforts securely and efficiently with:

- Enterprise-grade **security** and **access controls** including Single Sign-On (SSO).
- Granular **resource and token management** for better governance.
- Detailed **audit logs** and **usage analytics**.
- Private **data storage** and dataset viewing for enhanced collaboration.
- Advanced **compute options** including ZeroGPU quota boosts.
- Flexible **subscription plans** to suit teams and enterprises.

Enterprise customers benefit from dedicated support, customizable contracts, and a platform designed to support large-scale AI deployments.

---

## Community & Culture

- **Open & Ethical AI:** Committed to transparency, ethics, and open collaboration.
- **Inclusive & Supportive:** A fast-growing community where newcomers and experts alike share knowledge and work together.
- **Innovative:** Constantly pushing the boundaries with cutting-edge libraries, tools, and research.
- **Learner-Focused:** Build your machine learning profile by sharing projects and contributing to open source.

Join a global network of ML engineers and scientists shaping the future of AI.

---

## Careers at Hugging Face

Be part of the AI revolution! Hugging Face hires talented individuals passionate about machine learning, open source, and community building. Working here means contributing to impactful projects, collaborating with world-class teams, and helping build the tools that empower millions of users worldwide.

Explore current job openings and become a contributor to a mission-driven organization accelerating AI for good.

---

## Join the Future of AI

- **Sign up** for free and start exploring millions of models, datasets, and apps in seconds.
- Share your own projects and build a public portfolio to gain recognition.
- Collaborate with industry leaders, researchers, and a vibrant community dedicated to advancing machine learning.

Discover more at [huggingface.co](https://huggingface.co)

---

## Brand Colors & Assets

- Signature colors: Yellow (#FFD21E), Orange (#FF9D00), Grey (#6B7280)
- Available logos and brand materials in SVG, PNG, and AI formats for community use.

---

**Hugging Face** — The collaboration platform for the machine learning community, powering the AI revolution together.